# Cheatsheet: Logistic Regression & Decision Boundaries for Economics

A quick-reference companion to the Skeleton/Solution notebooks. Formulas, interpretation rules, code snippets, and pitfalls — no exercises.

## 1. Core formulas

| Concept | Formula |
|---|---|
| Linear score (log-odds), $k$ features | $z = \mathbf{w}\cdot\mathbf{x} + b = w_1x_1+\dots+w_kx_k+b$ |
| Sigmoid / logistic function | $g(z) = \dfrac{1}{1+e^{-z}}$ |
| Model prediction | $f_{\mathbf w,b}(\mathbf x) = g(\mathbf w\cdot\mathbf x + b) = \Pr(y=1\mid \mathbf x)$ |
| 0.5-threshold classification rule | predict $y=1$ if $\mathbf w\cdot\mathbf x + b \ge 0$, else $y=0$ |
| **Decision boundary** (2 features) | $w_0x_0+w_1x_1+b=0 \iff x_1 = -\dfrac{w_0x_0+b}{w_1}$ |
| Odds | $\text{odds} = \dfrac{f(\mathbf x)}{1-f(\mathbf x)} = e^{\mathbf w\cdot\mathbf x + b}$ |
| Log-odds (logit) | $\ln(\text{odds}) = \mathbf w\cdot\mathbf x + b$ — this is *linear* even though $f(\mathbf x)$ is not |
| Odds ratio for a 1-unit change in $x_j$ | $e^{w_j}$ (holding other features fixed) |
| Marginal effect on probability | $\dfrac{\partial f}{\partial x_j} = w_j\cdot f(\mathbf x)\bigl(1-f(\mathbf x)\bigr)$ — **not constant**, it depends on where you evaluate it |


## Setup (run once)

In [1]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    """Numerically stable sigmoid / logistic function."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def plot_data(X, y, ax, pos_label="Default (y=1)", neg_label="Repaid (y=0)", s=90):
    """Scatter-plot a 2-feature binary-outcome dataset."""
    y = np.array(y).reshape(-1)
    pos = y == 1
    neg = y == 0
    ax.scatter(X[pos, 0], X[pos, 1], marker='x', s=s, c='crimson', linewidths=2, label=pos_label)
    ax.scatter(X[neg, 0], X[neg, 1], marker='o', s=s, facecolors='none',
               edgecolors='steelblue', linewidths=2, label=neg_label)
    ax.legend(loc='best')

def draw_vthresh(ax, x):
    """Shade g(z)<0.5 vs g(z)>=0.5 either side of z=x on a sigmoid plot."""
    ylim, xlim = ax.get_ylim(), ax.get_xlim()
    ax.fill_between([xlim[0], x], ylim[1], alpha=0.15, color='steelblue')
    ax.fill_between([x, xlim[1]], ylim[1], alpha=0.15, color='crimson')
    ax.axvline(x, color='k', ls='--', lw=1)
    ax.set_xlim(xlim); ax.set_ylim(ylim)

def plot_linear_boundary(w, b, ax, x_range=(0, 4), color='seagreen', label='Decision boundary'):
    """Plot w0*x0 + w1*x1 + b = 0 and shade the predicted-0 region."""
    x0 = np.linspace(x_range[0], x_range[1], 200)
    x1 = -(w[0] * x0 + b) / w[1]
    ax.plot(x0, x1, c=color, lw=2, label=label)
    ax.fill_between(x0, x1, ax.get_ylim()[0], alpha=0.12, color=color)
    return x0, x1


## 2. Reading coefficients in economics terms

- **Sign of $w_j$**: direction of the relationship with the log-odds of the outcome. Positive $w_j$ → higher $x_j$ raises $\Pr(y=1)$; negative → lowers it.
- **Magnitude of $w_j$** is *not* directly a "percentage point" effect on probability (unlike OLS on a linear probability model). It's an effect on the **log-odds**. To talk about probability, either:
  - report the **odds ratio** $e^{w_j}$ ("a one-unit rise in DTI multiplies the odds of default by $e^{w_j}$"), or
  - report the **marginal effect at the mean** (or average marginal effect, AME) using the formula above.
- **The decision boundary line's slope** tells you the *rate of substitution* between two risk factors that keeps predicted risk constant: $\dfrac{dx_1}{dx_0}\Big|_{z=0} = -\dfrac{w_0}{w_1}$. This is the credit-risk analogue of an indifference curve — it says how much LTV must fall to offset a one-unit rise in DTI while holding default probability at 50%.
- **The intercept $b$** sets the *baseline* log-odds when all features are 0 — economically this is rarely a meaningful applicant, so it's mostly a normalizing constant, not something to over-interpret on its own.


## 3. Quick code snippets

### 3.1 Fit a logistic regression on real data (scikit-learn)
```python
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(X_train, y_train.ravel())

w = clf.coef_[0]      # array of length k
b = clf.intercept_[0] # scalar
print("weights:", w, "bias:", b)
print("odds ratios:", np.exp(w))
```

### 3.2 Plot a 2-feature linear decision boundary
```python
fig, ax = plt.subplots(figsize=(5,5))
plot_data(X, y, ax)
plot_linear_boundary(w, b, ax, x_range=(X[:,0].min(), X[:,0].max()))
ax.legend()
plt.show()
```

### 3.3 Convert coefficients to odds ratios with a table
```python
import pandas as pd
pd.DataFrame({"feature": feature_names, "coef (log-odds)": w, "odds_ratio": np.exp(w)})
```

### 3.4 Marginal effect at a specific point
```python
def marginal_effect(w, b, x, j):
    p = sigmoid(np.dot(w, x) + b)
    return w[j] * p * (1 - p)
```

### 3.5 Non-linear boundary via engineered features
```python
# e.g. add DTI^2 and an interaction term, then re-fit
X_poly = np.column_stack([X[:,0], X[:,1], X[:,0]**2, X[:,0]*X[:,1]])
clf.fit(X_poly, y.ravel())
```


## 4. Worked mini-example: threshold & boundary from parameters

Given $w_0=1,\ w_1=1,\ b=-3$ (mortgage-default example): boundary is $x_0+x_1=3$. Below is a reusable check you can copy-paste to sanity-check any new set of parameters.

In [2]:
def classify(w, b, x):
    "Return predicted class (0/1) and probability for a single point x."
    p = sigmoid(np.dot(w, x) + b)
    return int(p >= 0.5), p

w, b = np.array([1, 1]), -3
for point in [(0.5,1.5), (2,2), (3,0.5)]:
    cls, p = classify(w, b, point)
    print(f"x={point}  ->  P(default)={p:.3f}  ->  predicted class={cls}")


x=(0.5, 1.5)  ->  P(default)=0.269  ->  predicted class=0
x=(2, 2)  ->  P(default)=0.731  ->  predicted class=1
x=(3, 0.5)  ->  P(default)=0.622  ->  predicted class=1


## 5. When to reach for this approach (and when not to)

| Situation | Good fit? |
|---|---|
| Binary economic outcome (default/no default, recession/no recession, exit/survive) and you mainly care about *classification* and *marginal-probability* interpretation | ✅ Yes |
| You want coefficients interpretable as (approximate) probability effects directly, without transformation | ⚠️ Use a **linear probability model (LPM)** instead, or report AMEs |
| Outcome is ordered (credit rating categories) or a count | ❌ Use ordered logit/probit or a count model (Poisson/NB) instead |
| True relationship between features and log-odds is highly non-linear/non-monotonic | ⚠️ Add polynomial/interaction terms, or use a non-parametric classifier (trees, GBM) — see Limitations below |
| Small sample, many correlated regressors | ⚠️ Watch for perfect/quasi-separation and multicollinearity — MLE can fail to converge or produce huge, unstable coefficients |
| You need causal effects, not just predictive association | ❌ Logistic regression coefficients are associational; use quasi-experimental / IV / diff-in-diff methods for causal claims |

## 6. Common pitfalls
- **Confusing statistical significance with economic significance.** A tiny, "significant" coefficient can be economically negligible.
- **Reading $w_j$ as a probability effect directly.** It's a log-odds effect; convert via odds ratio or marginal effect.
- **Perfect separation.** If a boundary perfectly separates classes (as in our toy 6-point example!), MLE coefficients can diverge to $\pm\infty$ in real optimization — toy examples are illustrative, not a fitting recipe.
- **Class imbalance.** Rare economic events (defaults, crises) mean accuracy is a poor metric; use precision/recall, ROC-AUC, or a cost-weighted loss.
- **Extrapolating the boundary.** The line/curve is only trustworthy inside the range of the training data's features.
- **Ignoring reverse causality / simultaneity.** E.g. a bank may *change loan terms* in response to perceived risk, contaminating the estimated relationship.


## 7. Other economics use-cases with the same machinery

| Use case | Typical features (x) | Outcome (y) |
|---|---|---|
| Credit scoring / loan default | DTI, LTV, credit utilization, payment history | Default vs. repaid |
| Recession early-warning (probit is more common here) | Yield-curve slope, unemployment change, ISM index | Recession vs. no recession (next 12mo) |
| Bank failure prediction | Capital ratio, NPL ratio, liquidity ratio | Fails vs. survives |
| Sovereign debt crisis | Debt/GDP, reserves/imports, current account balance | Default/restructuring vs. no crisis |
| Labor force participation | Age, education, local wage, household income | In labor force vs. not |
| Firm bankruptcy (Altman-Z style) | Working capital/assets, retained earnings/assets, EBIT/assets | Bankrupt vs. solvent |
| Loan/insurance application approval | Income, existing debt, claims history | Approved vs. denied |
